# BCI Motor Imagery Project: Quick Overview
- **What we are doing**: Looking at brain waves when someone imagines moving their left or right hand.
- **Why we filter it (8–30 Hz)**: Brain signals are noisy. This filter cuts out the junk and keeps only the brain rhythms (mu and beta waves) that matter for hand movement.

## Section 1: Data Acquisition & Preprocessing
In this section, we load raw 64-channel EEG recordings (`.edf` files) corresponding to Motor Imagery tasks. 

To evaluate the effect of data scaling on classification accuracy, we dynamically concatenate multiple runs (`mne.concatenate_raws`) and apply an **8–30 Hz band-pass filter** to isolate Mu and Beta rhythms associated with motor imagery.

> **Demo Control:** Uncomment files below to scale training trials and observe real-time performance gains.

In [ ]:
from pathlib import Path
import mne

# Define your base file directory
DATA_DIR = Path.home() / "Desktop" / "files"

# Add or remove files here to show the judges accuracy changes! ---
run_files = [
    DATA_DIR / "S001" / "S001R04.edf",     
   
            
]

# Load and combine the file(s) automatically
raw_list = [mne.io.read_raw_edf(f, preload=True, verbose=False) for f in run_files]
if len(raw_list) > 1:
    raw = mne.concatenate_raws(raw_list)
else:
    raw = raw_list[0]

# Apply your 8–30 Hz band-pass filter to clean the noise
raw.filter(l_freq=8.0, h_freq=30.0, fir_design='firwin', verbose=False)

print(f"✅ Successfully loaded {len(run_files)} file(s) and filtered the data!")

## Section 2: EEG Signal Visualization
Here we plot the continuous, filtered 64-channel EEG signals. 

Visualizing the signals ensures artifact removal (e.g., baseline drift) and confirms that the band-pass filter successfully retains relevant spectral components across all combined runs before training the model.

In [ ]:
# 3. Plot the filtered EEG signals to get your graph for the paper
fig = raw.plot(duration=5, start=10, n_channels=8, scalings="auto", title="Filtered EEG Motor Imagery Signals")

## Section 3: Feature Extraction & Machine Learning Pipeline
This section implements a standard Brain-Computer Interface (BCI) classification pipeline:
1. **Epoching:** Extracting task-related intervals ($0.0 \text{s}$ to $3.0 \text{s}$) following event cues.
2. **Common Spatial Patterns (CSP):** Maximizing signal variance for one class while minimizing it for the other to extract spatial features across 64 channels.
3. **Linear Discriminant Analysis (LDA):** Fitting a decision boundary on CSP features to classify Left Hand vs. Right Hand motor imagery.
4. **Validation:** 5-fold cross-validation using `ShuffleSplit`.

### **Data Scaling Benchmark Results**
* **1 File (15 trials):** ~46.67% accuracy *(Baseline / Noise dominated)*
* **2 Files (30 trials):** ~56.67% accuracy *(Improved feature separation)*
* **3 Files (45 trials):** **80.00% accuracy** *(Optimal CSP convergence)*

In [ ]:
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import ShuffleSplit, cross_val_score
from sklearn.pipeline import Pipeline

# 1. Find the event markers (T1 = left fist, T2 = right fist)
events, event_id = mne.events_from_annotations(raw, verbose=False)
selected_events = {'T1': event_id['T1'], 'T2': event_id['T2']}

# 2. Chop data into 3-second trials for left vs. right imagination
epochs = mne.Epochs(raw, events, event_id=selected_events, tmin=0.0, tmax=3.0, baseline=None, preload=True, verbose=False)

X = epochs.get_data()  # The brain wave trials
y = epochs.events[:, -1]  # The labels (Left vs. Right)

# 3. Build Machine Learning Pipeline: CSP (Spatial filtering) + LDA (Classifier)
csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)
lda = LinearDiscriminantAnalysis()
clf = Pipeline([('CSP', csp), ('LDA', lda)])

# 4. Test the model using cross-validation
cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
scores = cross_val_score(clf, X, y, cv=cv, scoring='accuracy')

print(f"\n--- RESULTS FOR YOUR PAPER ---")
print(f"Mean Classification Accuracy: {scores.mean() * 100:.2f}%")